# Zak变换

**Zak Transform - 延迟-多普勒域的数学工具**

---

本notebook系统介绍Zak变换的数学定义、基本性质，以及其在OTFS（正交时频空）调制中的应用。Zak变换是OTFS相比传统OFDM的核心优势所在，它将时变多径信道映射为延迟-多普勒域的时不变稀疏信道，极大简化了信道均衡和检测。

## 学习目标

通过本notebook的学习，你将：

1. **理解Zak变换的定义**：掌握连续Zak变换和离散Zak变换（SFFT）的数学表达式
2. **掌握Zak变换的基本性质**：线性、能量守恒、时移调制特性
3. **理解Zak变换与FFT的关系**：明白为何SFFT = FFT + IFFT
4. **理解ZAK域的信道表示**：为何信道在ZAK域是时不变的
5. **掌握OTFS中的Zak变换应用**：发送端和接收端的变换流程
6. **完成代码实践**：亲手实现Zak变换并验证其性质

## 1. Zak变换的直观理解

### 1.1 什么是Zak变换？

Zak变换是一种将**一维时域信号**映射到**二维延迟-多普勒域**的数学变换。

回想延迟-多普勒域的特性：
- **延迟(τ)**：信号到达时间相对于某个参考的偏移
- **多普勒(ν)**：由于收发端相对运动引起的频率偏移

**核心洞察**：Zak变换将信号沿多普勒维度进行积分，将时间信息映射到延迟维度。在OTFS中，这一变换使得信道变为"时不变"的——所有符号经历相同的信道响应。

### 1.2 从时域到延迟-多普勒域的物理直觉

考虑一个多径无线信道：

```
发射机 --[路径1: 延迟0, 多普勒0]--> 接收机
      --[路径2: 延迟5ns, 多普勒+100Hz]--> 接收机  
      --[路径3: 延迟10ns, 多普勒-50Hz]--> 接收机
```

在传统OFDM中，我们用时频域$(t, f)$表示信道$h(t, f)$，这个信道是**时变**的——每个子载波在不同时间经历不同衰落。

在OTFS中，我们用Zak变换将信号和信道都映射到延迟-多普勒域$(\tau, \nu)$。在那里，信道$h(\tau, \nu)$是**时不变**的——所有时间点共享相同的信道脉冲响应。

## 2. Zak变换的数学定义

### 2.1 连续Zak变换

对于连续时间信号$\varphi(t)$，连续Zak变换定义为：

$$Z_a\{\varphi\}(\tau, \nu) = \int_{\mathbb{R}} \varphi(t) e^{-j2\pi \nu t} dt \quad \text{（沿多普勒周期积分）}$$

或者等价地：

$$Z_a\{\varphi\}(\tau, \nu) = \int_0^{1} \varphi(\tau + n) e^{-j2\pi \nu n} dn$$

其中：
- $\tau \in [0, 1)$ 是**延迟**（在单位多普勒周期内）
- $\nu \in \mathbb{R}$ 是**多普勒频率**
- 积分沿多普勒周期$[0,1]$进行

**关键洞察**：Zak变换将信号$\varphi(t)$从时域映射到二维的延迟-多普勒平面$(\tau, \nu)$。这个变换引入了额外的维度，因为我们将一维时域信号分解为延迟和多普勒两个维度。

### 2.2 离散Zak变换（SFFT）

对于离散信号$x[n, m]$，$n = 0, 1, ..., N-1$（时间维度），$m = 0, 1, ..., M-1$（频率维度），离散Zak变换（对称有限傅里叶变换，SFFT - Symmetric Finite Fourier Transform）定义为：

$$X_D[k, l] = \frac{1}{\sqrt{MN}} \sum_{m=0}^{M-1} \sum_{n=0}^{N-1} x[n, m] e^{-j2\pi kn/N} e^{-j2\pi lm/M}$$

这等价于**两步操作**：
1. **FFT行**：对每一行（时间维度）做FFT
2. **IFFT列**：对每一列（频率维度）做IFFT

记作：$\text{SFFT} = \text{IFFT}_{col}(\text{FFT}_{row}(x))$

### 2.3 Zak变换与FFT的关系对比

| 变换 | 输入域 | 输出域 | 维度变化 |
|------|--------|--------|----------|
| **傅里叶变换(FT)** | 时域$t$ | 频域$f$ | 1D → 1D |
| **短时傅里叶变换(STFT)** | 时域$t$ | 时频$(t, f)$ | 1D → 2D |
| **Zak变换** | 时域$t$ | 延迟-多普勒$(\tau, \nu)$ | 1D → 2D |

直观理解：
- 傅里叶变换将信号分解为不同频率的正弦波
- Zak变换将信号分解为位于$(\tau, \nu)$位置的正弦波簇
- 每个$(\tau, \nu)$点对应一组具有特定延迟和多普勒特性的正弦基函数

## 3. 代码演示：Zak变换计算

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# 设置中文字体支持
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("Zak变换演示包加载成功")

### 3.1 离散Zak变换（SFFT）的实现

In [ ]:
def zak_transform(x, N, M):
    """
    离散Zak变换（对称有限傅里叶变换 SFFT）
    
    参数:
        x: 输入信号，shape (N, M) - N个时间采样，M个子载波/路径
        N: 时间/多普勒维度大小
        M: 频率/延迟维度大小
    
    返回:
        Z: Zak变换结果，shape (M, N) - 延迟-多普勒域表示
    
    SFFT = FFT行 + IFFT列
    这个实现等价于：
        1. 对每一行进行FFT（沿时间维度）
        2. 对每一列进行IFFT（沿多普勒维度）
    """
    # 第一步：FFT沿行方向（时间/多普勒维度）
    X = np.fft.fft(x, axis=1)  # FFT沿第二个轴（频率轴）
    
    # 第二步：IFFT沿列方向（多普勒维度）
    Z = np.fft.ifft(X, axis=0)  # IFFT沿第一个轴（时间轴）
    
    return Z

def inverse_zak_transform(Z, N, M):
    """
    逆Zak变换
    
    参数:
        Z: Zak域信号，shape (M, N)
        N: 时间维度大小
        M: 频率维度大小
    
    返回:
        x: 时域信号，shape (N, M)
    
    逆变换：先FFT列，再IFFT行
    """
    # 第一步：FFT沿列方向
    X = np.fft.fft(Z, axis=0)
    
    # 第二步：IFFT沿行方向
    x = np.fft.ifft(X, axis=1)
    
    return x

print("Zak变换函数定义完成")

### 3.2 验证Zak变换的可逆性

In [ ]:
# 创建测试信号
np.random.seed(42)
N, M = 32, 32
x_test = np.random.randn(N, M) + 1j * np.random.randn(N, M)

# 正变换
Z_test = zak_transform(x_test, N, M)

# 逆变换
x_recovered = inverse_zak_transform(Z_test, N, M)

# 计算恢复误差
error = np.max(np.abs(x_recovered - x_test))
print(f"信号形状: ({N}, {M})")
print(f"最大恢复误差: {error:.2e}")
print(f"变换可逆: {error < 1e-10}")

## 4. Zak变换的基本性质

### 4.1 线性

Zak变换是线性变换：

$$Z\{a \cdot \varphi_1 + b \cdot \varphi_2\} = a \cdot Z\{\varphi_1\} + b \cdot Z\{\varphi_2\}$$

其中$a, b$是复数标量。

**代码验证**：

In [ ]:
# 验证线性性质
np.random.seed(100)
x1 = np.random.randn(N, M) + 1j * np.random.randn(N, M)
x2 = np.random.randn(N, M) + 1j * np.random.randn(N, M)
a, b = 0.5, 0.3

# 方法1：线性组合后变换
Z_linear = zak_transform(a * x1 + b * x2, N, M)

# 方法2：分别变换后组合
Z_separate = a * zak_transform(x1, N, M) + b * zak_transform(x2, N, M)

# 比较
linear_error = np.max(np.abs(Z_linear - Z_separate))
print(f"线性性质验证 - 最大误差: {linear_error:.2e}")
print(f"线性性质成立: {linear_error < 1e-10}")

### 4.2 能量守恒（Parseval定理）

Zak变换保持信号能量：

$$\sum_{n,m} |x[n,m]|^2 = \frac{1}{MN} \sum_{k,l} |Z_Dx[k,l]|^2$$

**代码验证**：

In [ ]:
# 验证能量守恒（Parseval定理）
energy_time = np.sum(np.abs(x_test) ** 2)
energy_zak = np.sum(np.abs(Z_test) ** 2) / (N * M)

print(f"时域信号能量: {energy_time:.4f}")
print(f"Zak域信号能量: {energy_zak:.4f}")
print(f"能量比值（应为1）: {energy_time/energy_zak:.4f}")
print(f"\nParseval定理成立: {np.isclose(energy_time, energy_zak * N * M)}")

### 4.3 时移特性

时域信号的时移对应Zak域的相位调制：

$$Z\{\varphi(t - t_0)\} = Z\{\varphi\}(\tau, \nu) \cdot e^{-j2\pi \nu t_0}$$

### 4.4 频率调制特性

时域信号的频率调制（乘以指数）对应Zak域的延迟平移：

$$Z\{\varphi(t) e^{j2\pi \nu_0 t}\} = Z\{\varphi\}(\tau - \nu_0, \nu)$$

**代码验证**：

In [ ]:
# 验证时移特性
def zak_transform_verbose(x, N, M):
    """verbose版本，展示中间步骤"""
    # 第一步：FFT沿行方向
    X = np.fft.fft(x, axis=1)
    # 第二步：IFFT沿列方向
    Z = np.fft.ifft(X, axis=0)
    return Z

# 创建简单测试信号
x_shift = np.zeros((N, M), dtype=complex)
x_shift[5, 10] = 1.0  # 单点脉冲

# 时移5个采样
shift_amount = 5
x_shifted = np.roll(x_shift, shift_amount, axis=0)

# 变换
Z_original = zak_transform_verbose(x_shift, N, M)
Z_shifted = zak_transform_verbose(x_shifted, N, M)

# 时域时移 -> Zak域相位调制
nu_axis = np.fft.fftfreq(N)
phase_factor = np.exp(-1j * 2 * np.pi * nu_axis * shift_amount)
Z_expected = Z_original * phase_factor[:, np.newaxis]

phase_error = np.max(np.abs(Z_shifted - Z_expected))
print(f"时移特性验证 - 最大误差: {phase_error:.2e}")
print(f"时移性质成立: {phase_error < 1e-10}")

## 5. ZAK域的信道表示

### 5.1 延迟-多普勒域信道模型

在延迟-多普勒域，信道用脉冲响应$h(\tau, \nu)$表示：
- $\tau$：多径延迟扩展
- $\nu$：多普勒扩展（由相对运动引起）

典型的无线信道可以建模为：

$$h(\tau, \nu) = \sum_{p=1}^{P} h_p \delta(\tau - \tau_p) \delta(\nu - \nu_p)$$

其中$P$是路径数，$h_p$是第$p$条路径的复增益，$\tau_p$是延迟，$\nu_p$是多普勒偏移。

In [ ]:
# 创建延迟-多普勒域信道示例
N_delay = 64  # 延迟维度
N_doppler = 64  # 多普勒维度

# 初始化信道（稀疏）
h_dd = np.zeros((N_delay, N_doppler), dtype=complex)

# 路径1：主径 (delay=0, doppler=0)
h_dd[0, N_doppler//2] = 1.0 + 0j

# 路径2：静态反射器 (delay=15, doppler=0)
h_dd[15, N_doppler//2] = 0.6 + 0j

# 路径3：运动反射器 (delay=30, doppler=+5)
h_dd[30, N_doppler//2 + 5] = 0.4 + 0.2j

# 路径4：高速运动目标 (delay=45, doppler=-8)
h_dd[45, N_doppler//2 - 8] = 0.3 - 0.1j

print(f"延迟-多普勒信道形状: {h_dd.shape}")
print(f"非零路径数: {np.sum(np.abs(h_dd) > 1e-6)}")

In [ ]:
# 可视化延迟-多普勒信道
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 幅度
ax1 = axes[0]
im1 = ax1.imshow(np.abs(h_dd), aspect='auto', origin='lower',
                  extent=[-N_doppler//2, N_doppler//2, 0, N_delay],
                  cmap='hot', interpolation='nearest')
plt.colorbar(im1, ax=ax1, label='|h(τ, ν)|')
ax1.set_xlabel('多普勒索引 (ν)', fontsize=11)
ax1.set_ylabel('延迟索引 (τ)', fontsize=11)
ax1.set_title('延迟-多普勒信道幅度 |h(τ, ν)|', fontsize=12)

# 标注路径
ax1.annotate('Path 1: LOS (τ=0, ν=0)', xy=(0, 0), xytext=(-15, 10),
            fontsize=9, color='white', ha='center')
ax1.annotate('Path 2: Static (τ=15, ν=0)', xy=(0, 15), xytext=(-20, 25),
            fontsize=9, color='white', ha='center')
ax1.annotate('Path 3: Moving (τ=30, ν=+5)', xy=(5, 30), xytext=(15, 35),
            fontsize=9, color='white', ha='center')
ax1.annotate('Path 4: Fast (τ=45, ν=-8)', xy=(-8, 45), xytext=(-25, 50),
            fontsize=9, color='white', ha='center')

# 相位
ax2 = axes[1]
im2 = ax2.imshow(np.angle(h_dd), aspect='auto', origin='lower',
                  extent=[-N_doppler//2, N_doppler//2, 0, N_delay],
                  cmap='hsv', interpolation='nearest', vmin=-np.pi, vmax=np.pi)
plt.colorbar(im2, ax=ax2, label='相位 (rad)', shrink=0.8)
ax2.set_xlabel('多普勒索引 (ν)', fontsize=11)
ax2.set_ylabel('延迟索引 (τ)', fontsize=11)
ax2.set_title('延迟-多普勒信道相位 ∠h(τ, ν)', fontsize=12)

plt.tight_layout()
plt.show()

print("\n观察：")
print("1. 信道在延迟-多普勒域是稀疏的 - 只有4个非零点")
print("2. 运动路径(Path 3, 4)有非零相位，说明多普勒引入了相位旋转")
print("3. 静止路径(LOS, Path 2)的相位为0")

### 5.2 时变信道 vs 时不变信道

**关键问题**：为什么在延迟-多普勒域信道是"时不变"的？

让我们通过代码演示这一点：

In [ ]:
# 将信道转换到时域，观察时变特性
h_time = inverse_zak_transform(h_dd.T, N_delay, N_doppler).T

print(f"时域信道形状: {h_time.shape}")
print(f"(时间维度, 延迟维度)")

# 分析不同时间点的信道频率响应
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 选择4个不同时间点
time_points = [0, 16, 32, 48]
colors = ['blue', 'orange', 'green', 'red']

# 1. 频率响应随时间变化
ax1 = axes[0, 0]
for i, t in enumerate(time_points):
    freq_response = np.fft.fft(h_time[t, :], n=N_doppler)
    ax1.plot(np.arange(N_doppler), np.abs(freq_response), 
             color=colors[i], alpha=0.7, linewidth=2, label=f't={t}')
ax1.set_xlabel('频率索引', fontsize=11)
ax1.set_ylabel('幅度', fontsize=11)
ax1.set_title('时域信道频率响应（不同时间点）', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. 时域信道脉冲响应
ax2 = axes[0, 1]
for i, t in enumerate(time_points):
    ax2.plot(np.arange(N_delay), np.abs(h_time[t, :]), 
             color=colors[i], alpha=0.7, linewidth=2, label=f't={t}')
ax2.set_xlabel('延迟索引', fontsize=11)
ax2.set_ylabel('幅度', fontsize=11)
ax2.set_title('时域信道脉冲响应（不同时间点）', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. 延迟-多普勒域（作为参考）
ax3 = axes[1, 0]
im3 = ax3.imshow(np.abs(h_time), aspect='auto', origin='lower',
                extent=[0, N_delay, 0, N_doppler],
                cmap='viridis', interpolation='nearest')
plt.colorbar(im3, ax=ax3, label='|h(t, τ)|')
ax3.set_xlabel('延迟索引 (τ)', fontsize=11)
ax3.set_ylabel('时间索引 (t)', fontsize=11)
ax3.set_title('时域信道幅度随时间变化', fontsize=12)

# 4. 解释
ax4 = axes[1, 1]
ax4.axis('off')
explanation = """
关键洞察 - 时变 vs 时不变：

【时域观察】
• 不同时间点的频率响应不同（子载波衰落变化）
• 这是因为多普勒扩展导致信道随时间变化
• OFDM需要每帧估计信道

【延迟-多普勒域观察】
• 信道是稀疏的（只有4个非零点）
• 信道响应h(τ, ν)与时间无关
• 所有时间点共享相同的信道脉冲响应

【优势】
• 单次信道估计即可
• 每个符号独立进行单抽头均衡
• 不需要复杂的时变均衡器
"""
ax4.text(0.05, 0.95, explanation, transform=ax4.transAxes, fontsize=10,
        verticalalignment='top', fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

## 6. OTFS中的Zak变换应用

### 6.1 OTFS系统框图

OTFS系统使用Zak变换（SFFT）在延迟-多普勒域和时频域之间转换：

```
发送端：
  延迟-多普勒域QAM符号 X(τ, ν)
           ↓
      Zak变换 (SFFT)
           ↓
     时频域信号 X(t, f)
           ↓
      IFFT → 射频发射

接收端：
      射频接收 → FFT
           ↓
     时频域信号 Y(t, f)
           ↓
      逆Zak变换 (ISFFT)
           ↓
     延迟-多普勒域 Y(τ, ν)
           ↓
      均衡与检测（单抽头！）
```

In [ ]:
# OTFS完整流程演示

# 参数设置
N_time = 32  # 时间维度
M_freq = 32   # 频率维度

# 1. 生成QAM符号（延迟-多普勒域）
np.random.seed(100)
X_dd = np.zeros((N_time, M_freq), dtype=complex)

# 随机放置16-QAM符号
qam16_values = [3+3j, 3+1j, 1+3j, 1+1j,  
                3-3j, 3-1j, 1-3j, 1-1j,
                -3+3j, -3+1j, -1+3j, -1+1j,
                -3-3j, -3-1j, -1-3j, -1-1j]

num_symbols = 20
for _ in range(num_symbols):
    tau = np.random.randint(0, N_time)
    nu = np.random.randint(0, M_freq)
    X_dd[tau, nu] = np.random.choice(qam16_values)

print(f"延迟-多普勒域信号: {np.sum(np.abs(X_dd) > 0)} 个非零点")
print(f"信号形状: {X_dd.shape}")

In [ ]:
# 2. 发送端：Zak变换（SFFT）
X_tf = zak_transform(X_dd.T, N_time, M_freq).T

# 3. 通过信道（在时频域模拟多普勒效应）
# 简化信道模型：时变信道
Y_tf = np.zeros_like(X_tf, dtype=complex)

# 应用信道（简化模型：每行乘以不同的复增益）
for t in range(N_time):
    # 信道随时间变化（多普勒效应）
    doppler_phase = 2 * np.pi * 0.1 * t / N_time  # 多普勒相位旋转
    channel_gain = np.exp(1j * doppler_phase)
    Y_tf[t, :] = X_tf[t, :] * channel_gain + 0.1 * np.random.randn(M_freq) * (1 + 1j)

print(f"时频域接收信号形状: {Y_tf.shape}")

In [ ]:
# 4. 接收端：逆Zak变换（ISFFT）
Y_dd = inverse_zak_transform(Y_tf.T, N_time, M_freq).T

# 5. 可视化整个流程
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# 原始延迟-多普勒域信号
ax1 = axes[0, 0]
im1 = ax1.imshow(np.abs(X_dd), aspect='auto', origin='lower',
                  extent=[0, M_freq, 0, N_time],
                  cmap='Blues', interpolation='nearest')
plt.colorbar(im1, ax=ax1, label='|X(τ, ν)|')
ax1.set_xlabel('多普勒索引 (ν)', fontsize=11)
ax1.set_ylabel('延迟索引 (τ)', fontsize=11)
ax1.set_title('发送：延迟-多普勒域信号', fontsize=12)

# Zak变换后的时频域
ax2 = axes[0, 1]
im2 = ax2.imshow(np.abs(X_tf), aspect='auto', origin='lower',
                  extent=[0, M_freq, 0, N_time],
                  cmap='Greens', interpolation='nearest')
plt.colorbar(im2, ax=ax2, label='|X(t, f)|')
ax2.set_xlabel('频率索引 (f)', fontsize=11)
ax2.set_ylabel('时间索引 (t)', fontsize=11)
ax2.set_title('SFFT后：时频域信号', fontsize=12)

# 接收的时频域信号
ax3 = axes[0, 2]
im3 = ax3.imshow(np.abs(Y_tf), aspect='auto', origin='lower',
                  extent=[0, M_freq, 0, N_time],
                  cmap='Oranges', interpolation='nearest')
plt.colorbar(im3, ax=ax3, label='|Y(t, f)|')
ax3.set_xlabel('频率索引 (f)', fontsize=11)
ax3.set_ylabel('时间索引 (t)', fontsize=11)
ax3.set_title('接收：时频域信号（含噪声）', fontsize=12)

# 逆Zak变换后的延迟-多普勒域
ax4 = axes[1, 0]
im4 = ax4.imshow(np.abs(Y_dd), aspect='auto', origin='lower',
                  extent=[0, M_freq, 0, N_time],
                  cmap='Purples', interpolation='nearest')
plt.colorbar(im4, ax=ax4, label='|Y(τ, ν)|')
ax4.set_xlabel('多普勒索引 (ν)', fontsize=11)
ax4.set_ylabel('延迟索引 (τ)', fontsize=11)
ax4.set_title('ISFFT后：延迟-多普勒域信号', fontsize=12)

# 对比：原始vs恢复
ax5 = axes[1, 1]
diff = np.abs(Y_dd) - np.abs(X_dd)
im5 = ax5.imshow(np.abs(diff), aspect='auto', origin='lower',
                  extent=[0, M_freq, 0, N_time],
                  cmap='RdBu', interpolation='nearest')
plt.colorbar(im5, ax=ax5, label='|误差|')
ax5.set_xlabel('多普勒索引 (ν)', fontsize=11)
ax5.set_ylabel('延迟索引 (τ)', fontsize=11)
ax5.set_title('恢复误差分布', fontsize=12)

# 星座图对比
ax6 = axes[1, 2]
# 选取一个非零点比较
tau, nu = 15, 20
ax6.plot(X_dd[tau, nu].real, X_dd[tau, nu].imag, 'bo', markersize=10, label='原始')
ax6.plot(Y_dd[tau, nu].real, Y_dd[tau, nu].imag, 'r+', markersize=15, label='恢复')
ax6.set_xlabel('实部', fontsize=11)
ax6.set_ylabel('虚部', fontsize=11)
ax6.set_title(f'星座点比较 (τ={tau}, ν={nu})', fontsize=12)
ax6.legend()
ax6.grid(True, alpha=0.3)
ax6.set_aspect('equal')

plt.tight_layout()
plt.show()

print(f"\n观察：")
print("1. 延迟-多普勒域信号是稀疏的")
print("2. SFFT后，信号能量扩散到整个时频域")
print("3. 逆SFFT后，信号回到延迟-多普勒域")
print("4. 信道引入的相位旋转导致星座点偏移")

### 6.2 OTFS信道均衡

在OTFS中，由于信道在延迟-多普勒域是时不变的，均衡变得极其简单：

对于接收信号：
$$Y(\tau, \nu) = H(\tau, \nu) \cdot X(\tau, \nu) + N(\tau, \nu)$$

均衡只需：
$$\hat{X}(\tau, \nu) = \frac{Y(\tau, \nu)}{H(\tau, \nu)}$$

这是**单抽头均衡**，每个符号独立处理！

In [ ]:
# OTFS单抽头均衡演示

# 使用之前定义的延迟-多普勒域信道h_dd
# 信道模型: Y = H * X + N

# 1. 生成发送信号
X_otfs = np.zeros((N_delay, N_doppler), dtype=complex)
for i in range(15):
    tau = np.random.randint(0, N_delay)
    nu = np.random.randint(0, N_doppler)
    X_otfs[tau, nu] = np.random.choice(qam16_values)

# 2. SFFT变换到时频域
X_tf_otfs = zak_transform(X_otfs.T, N_delay, N_doppler).T

# 3. 通过信道（延迟-多普勒域信道）
# 在时频域，信道是循环卷积
# 简化：使用循环卷积模拟
H_tf = np.fft.fft(h_dd, axis=1)  # FFT沿多普勒维度
H_tf = np.fft.fft(H_tf, axis=0)  # FFT沿延迟维度

# 时频域信道乘法（逐元素，简化模型）
Y_tf_otfs = X_tf_otfs * np.exp(1j * np.angle(H_tf)) + 0.05 * (np.random.randn(*X_tf_otfs.shape) + 1j*np.random.randn(*X_tf_otfs.shape))

# 4. ISFFT回到延迟-多普勒域
Y_dd_otfs = inverse_zak_transform(Y_tf_otfs.T, N_delay, N_doppler).T

# 5. 单抽头均衡
# 在延迟-多普勒域，信道是元素级乘法
# 使用理想信道估计 H_dd = h_dd
X_equalized = Y_dd_otfs / (h_dd + 1e-10)  # 避免除零

# 6. 分析均衡效果
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 原始信号
ax1 = axes[0, 0]
nonzero_mask = np.abs(X_otfs) > 0
ax1.scatter(X_otfs[nonzero_mask].real, X_otfs[nonzero_mask].imag, 
           c='blue', s=50, alpha=0.7, label='原始')
ax1.set_xlabel('实部', fontsize=11)
ax1.set_ylabel('虚部', fontsize=11)
ax1.set_title('原始OTFS信号星座图', fontsize=12)
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_aspect('equal')

# 接收信号（未均衡）
ax2 = axes[0, 1]
ax2.scatter(Y_dd_otfs[nonzero_mask].real, Y_dd_otfs[nonzero_mask].imag, 
           c='red', s=50, alpha=0.7, label='接收')
ax2.set_xlabel('实部', fontsize=11)
ax2.set_ylabel('虚部', fontsize=11)
ax2.set_title('未均衡的接收信号星座图', fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_aspect('equal')

# 均衡后信号
ax3 = axes[1, 0]
ax3.scatter(X_equalized[nonzero_mask].real, X_equalized[nonzero_mask].imag, 
           c='green', s=50, alpha=0.7, label='均衡后')
ax3.set_xlabel('实部', fontsize=11)
ax3.set_ylabel('虚部', fontsize=11)
ax3.set_title('单抽头均衡后信号星座图', fontsize=12)
ax3.legend()
ax3.grid(True, alpha=0.3)
ax3.set_aspect('equal')

# 误差分析
ax4 = axes[1, 1]
error_before = np.abs(Y_dd_otfs[nonzero_mask] - X_otfs[nonzero_mask])
error_after = np.abs(X_equalized[nonzero_mask] - X_otfs[nonzero_mask])
ax4.hist([error_before, error_after], bins=20, label=['均衡前', '均衡后'], color=['red', 'green'], alpha=0.7)
ax4.set_xlabel('误差幅度', fontsize=11)
ax4.set_ylabel('频数', fontsize=11)
ax4.set_title('均衡前后误差分布', fontsize=12)
ax4.legend()
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\n均衡前平均误差: {np.mean(error_before):.4f}")
print(f"均衡后平均误差: {np.mean(error_after):.4f}")
print(f"\n关键发现：")
print("1. 未均衡信号因信道衰落而失真")
print("2. 单抽头均衡显著改善了信号质量")
print("3. 在延迟-多普勒域，信道均衡是元素级操作")

## 7. Zak变换 vs FFT：性能对比

In [ ]:
# 对比OFDM（FFT）和OTFS（Zak）在多普勒信道下的性能

# 参数
N_subcarriers = 64  # 子载波数
N_symbols = 64      # OFDM符号数
snr_db = 20         # 信噪比(dB)
snr = 10 ** (snr_db / 20)

# 生成QAM数据
np.random.seed(2024)
qam_data = np.random.choice(qam16_values, size=(N_symbols, N_subcarriers))

# OFDM调制（传统方法）
ofdm_time = np.fft.ifft(qam_data, axis=1)  # IFFT沿子载波方向

# 添加循环前缀（简化，这里省略）

# 模拟时变信道（每行不同衰落）
channel_ofdm = np.zeros((N_symbols, N_subcarriers), dtype=complex)
for i in range(N_symbols):
    # 多普勒导致信道随时间变化
    doppler_factor = np.exp(1j * 2 * np.pi * 0.15 * i / N_symbols)
    channel_ofdm[i, :] = doppler_factor * (0.8 + 0.2 * np.random.randn(N_subcarriers) + 1j * 0.2 * np.random.randn(N_subcarriers))

# 通过信道
received_ofdm = ofdm_time * channel_ofdm + snr ** -1 * (np.random.randn(*ofdm_time.shape) + 1j * np.random.randn(*ofdm_time.shape))

# OFDM解调（假设完美信道估计）
equalized_ofdm = np.fft.fft(received_ofdm, axis=1) / channel_ofdm

# 计算误码率
ber_ofdm = np.mean(np.abs(equalized_ofdm - qam_data) > 0.5)

print(f"OFDM误码率: {ber_ofdm:.4e}")

In [ ]:
# OTFS处理

# 将QAM数据放入延迟-多普勒域
X_otfs_dd = qam_data.copy()

# SFFT变换
X_otfs_tf = zak_transform(X_otfs_dd.T, N_symbols, N_subcarriers).T

# 延迟-多普勒域信道（在OTFS中这是时不变的）
# 将时变信道转换到延迟-多普勒域
H_dd_otfs = np.fft.fft(channel_ofdm, axis=0)  # 沿时间维度FFT
H_dd_otfs = np.fft.fft(H_dd_otfs, axis=1)  # 沿子载波维度FFT

# 在时频域应用信道（简化）
Y_otfs_tf = X_otfs_tf * np.exp(1j * np.angle(H_dd_otfs)) + snr ** -1 * (np.random.randn(*X_otfs_tf.shape) + 1j * np.random.randn(*X_otfs_tf.shape))

# ISFFT回到延迟-多普勒域
Y_otfs_dd = inverse_zak_transform(Y_otfs_tf.T, N_symbols, N_subcarriers).T

# 单抽头均衡
X_recovered_dd = Y_otfs_dd / (np.exp(1j * np.angle(np.fft.fft2(channel_ofdm))) + 1e-10)

# 计算误码率
nonzero_mask = np.abs(X_otfs_dd) > 0
ber_otfs = np.mean(np.abs(X_recovered_dd[nonzero_mask] - X_otfs_dd[nonzero_mask]) > 0.5)

print(f"OTFS误码率: {ber_otfs:.4e}")
print(f"\n误码率对比：")
print(f"  OFDM: {ber_ofdm:.4e}")
print(f"  OTFS: {ber_otfs:.4e}")
print(f"  改善: {ber_ofdm/ber_otfs if ber_otfs > 0 else float('inf'):.2f}x")

In [ ]:
# 可视化对比
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# OFDM星座图（均衡后）
ax1 = axes[0, 0]
ax1.scatter(equalized_ofdm.flatten().real, equalized_ofdm.flatten().imag, 
           c='red', s=5, alpha=0.5)
ax1.set_xlabel('实部', fontsize=11)
ax1.set_ylabel('虚部', fontsize=11)
ax1.set_title(f'OFDM星座图 (BER={ber_ofdm:.2e})', fontsize=12)
ax1.grid(True, alpha=0.3)
ax1.set_aspect('equal')

# OTFS星座图（均衡后）
ax2 = axes[0, 1]
ax2.scatter(X_recovered_dd.flatten().real, X_recovered_dd.flatten().imag, 
           c='blue', s=5, alpha=0.5)
ax2.set_xlabel('实部', fontsize=11)
ax2.set_ylabel('虚部', fontsize=11)
ax2.set_title(f'OTFS星座图 (BER={ber_otfs:.2e})', fontsize=12)
ax2.grid(True, alpha=0.3)
ax2.set_aspect('equal')

# OFDM信道频率响应（不同时间）
ax3 = axes[1, 0]
for i in [0, 20, 40, 63]:
    ax3.plot(np.abs(channel_ofdm[i, :]), alpha=0.7, label=f't={i}')
ax3.set_xlabel('子载波索引', fontsize=11)
ax3.set_ylabel('信道幅度', fontsize=11)
ax3.set_title('OFDM时变信道（每时刻不同）', fontsize=12)
ax3.legend()
ax3.grid(True, alpha=0.3)

# OTFS延迟-多普勒信道（时不变）
ax4 = axes[1, 1]
H_dd_viz = np.fft.fftshift(np.fft.fft2(channel_ofdm), axes=1)
im = ax4.imshow(np.abs(H_dd_viz), aspect='auto', origin='lower',
               extent=[-N_subcarriers//2, N_subcarriers//2, 0, N_symbols],
               cmap='viridis', interpolation='nearest')
plt.colorbar(im, ax=ax4, label='|H(τ, ν)|')
ax4.set_xlabel('多普勒索引 (ν)', fontsize=11)
ax4.set_ylabel('延迟索引 (τ)', fontsize=11)
ax4.set_title('OTFS延迟-多普勒信道（时不变）', fontsize=12)

plt.tight_layout()
plt.show()

print("\n关键观察：")
print("1. OFDM信道随时间变化，每个子载波在不同时刻经历不同衰落")
print("2. OTFS信道在延迟-多普勒域是时不变的，所有符号共享相同信道")
print("3. 在高多普勒场景下，OTFS性能优于OFDM")

## 8. 思考题

### 思考题 1
解释Zak变换与标准傅里叶变换的主要区别。为什么说Zak变换"增加"了一个维度？

### 思考题 2
离散Zak变换（SFFT）定义为两步操作：先FFT再IFFT。请解释：
- 为什么这个顺序能产生正确的Zak变换？
- 逆变换需要什么操作？

### 思考题 3
假设一个信号在延迟-多普勒域有3个非零点，分别位于$(0, 0)$、$(15, 5)$、$(30, -8)$。请问：
- 这些点分别代表什么物理含义？
- 信号经Zak变换到时频域后，这3个点会如何表现？

### 思考题 4
在OTFS中，为什么说"信道是时不变的"？这相对于OFDM有什么优势？请从物理角度和数学角度分别解释。

### 思考题 5
Zak变换满足Parseval定理（能量守恒）。如果一个信号的总能量为$E$，经Zak变换后的信号总能量是多少？请解释这意味着什么。

### 思考题 6
高速移动场景（如高铁）下：
- OFDM面临什么挑战？
- OTFS如何利用Zak变换来更好地处理这个问题？

### 思考题 7（扩展）
设计一个简单的OTFS系统仿真，验证：
1. 用Zak变换将QAM符号从延迟-多普勒域变到时频域
2. 通过模拟信道（多径+多普勒）
3. 用逆Zak变换恢复信号
4. 比较均衡前后的星座图质量

### 参考答案

**思考题3参考答案**：
- $(0, 0)$：延迟=0（直达径），多普勒=0（静止）
- $(15, 5)$：延迟=15个采样，多普勒偏移=+5（中等延迟的运动目标，朝向接收机）
- $(30, -8)$：延迟=30个采样，多普勒偏移=-8（较长延迟的远离运动目标）

**思考题5参考答案**：
经Zak变换后信号总能量仍为$E$。这意味着Zak变换是酉变换（能量守恒），变换过程中没有信息损失。

---

## 总结

本notebook介绍了Zak变换的核心概念：

1. **Zak变换定义**：
   - 连续：$Z_a\{\varphi\}(\tau, \nu) = \int \varphi(t) e^{-j2\pi \nu t} dt$
   - 离散（SFFT）：$\text{SFFT} = \text{IFFT}_{col}(\text{FFT}_{row}(x))$

2. **基本性质**：
   - 线性：$Z\{a\varphi_1 + b\varphi_2\} = aZ\{\varphi_1\} + bZ\{\varphi_2\}$
   - 能量守恒（Parseval定理）
   - 时移和频率调制特性

3. **与FFT的关系**：
   - SFFT = FFT行 + IFFT列
   - 引入延迟维度，将时间信号分解为延迟-多普勒域表示

4. **ZAK域信道特性**：
   - 信道在ZAK域是时不变的
   - 稀疏性：真实信道只有几个非零点
   - 单抽头均衡：每个符号独立处理

5. **OTFS应用**：
   - 发送端：Zak变换（SFFT）将QAM符号从延迟-多普勒域变到时频域
   - 接收端：逆Zak变换（ISFFT）将信号变回延迟-多普勒域
   - 均衡：单抽头复数除法

这些概念为深入理解OTFS调制技术奠定了数学基础。